# Download data

In [ ]:
!pip install -q torch torchvision scikit-learn matplotlib pillow tqdm huggingface_hub
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Output save
OUTPUT_DIR = '/content/drive/MyDrive/bollin_pollution'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
from google.colab import userdata
import zipfile, shutil

hf_token = userdata.get('HF_TOKEN')

DATA_DIR    = Path('/content/dataset')
IMGS_DIR    = DATA_DIR / 'imgs'
LABELS_FILE = DATA_DIR / 'pollution_incidents.txt'

if IMGS_DIR.exists() and any(IMGS_DIR.glob('*.jpg')):
    print('Dataset already extracted, skipping download.')
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading files from HuggingFace …')

    # Download the ZIP
    zip_path = hf_hub_download(
        repo_id='MarkP1929/mill-st-imgs',
        filename='imgs.zip',
        repo_type='dataset',
        local_dir=str(DATA_DIR),
        token=hf_token
    )

    # Download the labels
    hf_hub_download(
        repo_id='MarkP1929/mill-st-imgs',
        filename='pollution_incidents.txt',
        repo_type='dataset',
        local_dir=str(DATA_DIR),
        token=hf_token
    )

    print(f'Downloaded to {zip_path} — extracting …')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)
    print('Extraction complete')

all_images = sorted(DATA_DIR.rglob('*.jpg')) + sorted(DATA_DIR.rglob('*.png'))
print(f'Total images found: {len(all_images)}')

# Read positive labels
if LABELS_FILE.exists():
    with open(LABELS_FILE) as f:
        positive_set = {line.strip() for line in f if line.strip()}
    print(f'Pollution incidents labelled: {len(positive_set)}')
    if len(all_images) > 0:
        ratio = len(all_images) // max(len(positive_set), 1)
        print(f'Class ratio  - 1 positive per {ratio} images')
else:
    print(f'Error: {LABELS_FILE} not found!')

# Import config

In [5]:
# Imports and config
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import numpy as np
from pathlib import Path

from config import *
from dataset_utils import split_list, RiverDataset, SimpleDataset, make_weighted_sampler
from focal_loss import FocalLoss
from autoencoder import ConvAutoencoder
from train_autoencoder import train_autoencoder

# Split data

In [6]:
pos_paths = [p for p in all_images if p.name in positive_set]
neg_paths = [p for p in all_images if p.name not in positive_set]

pos_train, pos_val, pos_test = split_list(pos_paths, VAL_SPLIT, TEST_SPLIT)
neg_train, neg_val, neg_test = split_list(neg_paths, VAL_SPLIT, TEST_SPLIT)

train_paths = pos_train + neg_train
val_paths   = pos_val   + neg_val
test_paths  = pos_test  + neg_test
random.shuffle(train_paths)

# Train autoencoder

In [ ]:
clean_train = [p for p in train_paths if p.name not in positive_set]
clean_val   = [p for p in val_paths   if p.name not in positive_set]

ae_save = f'{OUTPUT_DIR}/autoencoder.pt'
ae_model, ae_train_hist, ae_val_hist = train_autoencoder(
    clean_train, clean_val, ae_save
)

# Plot loss
plt.figure(figsize=(8, 4))
plt.plot(ae_train_hist, label='Train MSE')
plt.plot(ae_val_hist, label='Val MSE')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Autoencoder Training')
plt.legend(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ae_loss.png', dpi=150)
plt.show()

# Train EfficientNet Classifier

In [ ]:
import torch
from torch.utils.data import DataLoader  #
from classifier import build_efficientnet, train_classifier, evaluate_classifier
from dataset_utils import RiverDataset, make_weighted_sampler

# Create datasets
train_dataset = RiverDataset(train_paths, positive_set, augment=True)
val_dataset   = RiverDataset(val_paths, positive_set, augment=False)
test_dataset  = RiverDataset(test_paths, positive_set, augment=False)

# Weighted sampler for training (balance positives)
train_sampler = make_weighted_sampler(train_paths, positive_set)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Build and train
clf_model = build_efficientnet(num_classes=1, pretrained=True)
clf_model, clf_train_loss, clf_val_loss = train_classifier(clf_model, train_loader, val_loader)

# Save model
torch.save(clf_model.state_dict(), f'{OUTPUT_DIR}/classifier.pt')

# Evaluate
clf_metrics = evaluate_classifier(clf_model, test_loader)
print("Classifier Test Metrics:")
print(f"AUC: {clf_metrics['auc']:.4f}")
print(f"Avg Precision: {clf_metrics['avg_precision']:.4f}")
print(clf_metrics['classification_report'])

In [ ]:
from visualization import plot_training_history, plot_pr_curve, plot_roc_curve, plot_reconstructions
from dataset_utils import SimpleDataset, AE_TF

# Classifier training curves
plot_training_history(clf_train_loss, clf_val_loss, title='Classifier Training',
                      save_path=f'{OUTPUT_DIR}/clf_loss.png')

# PR curve for classifier
plot_pr_curve(clf_metrics['labels'], clf_metrics['probs'],
              title='Classifier PR Curve', save_path=f'{OUTPUT_DIR}/clf_pr.png')

# Autoencoder reconstructions
ae_dataset = SimpleDataset(clean_val[:10], AE_TF)  # sample clean images
plot_reconstructions(ae_model, ae_dataset, DEVICE, num_samples=5,
                     save_path=f'{OUTPUT_DIR}/ae_reconstructions.png')

# Ensemble

In [ ]:
from ensemble import evaluate_ensemble

# Load models if needed
ae_model.load_state_dict(torch.load(f'{OUTPUT_DIR}/autoencoder.pt'))
clf_model.load_state_dict(torch.load(f'{OUTPUT_DIR}/classifier.pt'))

ensemble_scores, true_labels, ens_metrics, clf_m, ae_errs = evaluate_ensemble(
    ae_model, clf_model, test_paths, positive_set, DEVICE
)

print("Ensemble Test Metrics:")
print(f"AUC: {ens_metrics['auc']:.4f}")
print(f"Avg Precision: {ens_metrics['avg_precision']:.4f}")

# Plot ensemble PR curve
plot_pr_curve(true_labels, ensemble_scores, title='Ensemble PR Curve',
              save_path=f'{OUTPUT_DIR}/ensemble_pr.png')

# Inference on new sample

In [ ]:
from inference import load_models, predict_single_image, live_feed

# Load models
ae, clf = load_models(f'{OUTPUT_DIR}/autoencoder.pt', f'{OUTPUT_DIR}/classifier.pt')

# Test on a single image
sample_img = Image.open(test_paths[0])  # or any image
result = predict_single_image(ae, clf, sample_img)
print(result)

# Live monitoring

In [ ]:
from live_monitor import run_live_monitor

# Set thresholds (computed from validation data)
AE_THRESHOLD = 0.0008
CLF_THRESHOLD = 0.5

# Run the live monitor (this will run indefinitely until interrupted)
run_live_monitor(
    ae_model=ae_model,
    clf_model=clf_model,
    webcam_url="http://www.maccinfo.com/WebcamUtility/Utility.jpg",
    poll_interval=60,
    ae_threshold=AE_THRESHOLD,
    clf_threshold=CLF_THRESHOLD,
    base_dir=OUTPUT_DIR,
    device=DEVICE,
)